# Fine-tune FLAN-T5-base — Multi-Engine DB Log Analyzer

Trains `google/flan-t5-base` from scratch on a combined Oracle + PostgreSQL + MySQL log-analysis dataset.

**Before running:** upload `train_v2.jsonl` and `val_v2.jsonl` as a Kaggle dataset and update `DATASET_PATH` below.

**Expected training time:** ~3–5 hours on a Kaggle T4 GPU for 3 epochs on 86k examples.

In [ ]:
# ── Install / upgrade deps ──────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'transformers>=4.48.0', 'huggingface_hub<0.29.0', 'datasets', 'accelerate', 'sentencepiece'], check=True)
print('Dependencies ready.')


In [ ]:
import os

# ── CONFIGURE THESE PATHS ────────────────────────────────────────────────
# Point to wherever you uploaded the jsonl files as a Kaggle dataset.
# Example: if your dataset is named 'db-log-finetune', the path is:
#   /kaggle/input/db-log-finetune/
DATASET_PATH = '/kaggle/input/YOUR-DATASET-NAME'   # <-- CHANGE THIS TO YOUR DATASET NAME

TRAIN_FILE   = os.path.join(DATASET_PATH, 'train_v2.jsonl')
VAL_FILE     = os.path.join(DATASET_PATH, 'val_v2.jsonl')

# Starting from raw FLAN-T5-base (no local checkpoint needed)
BASE_MODEL   = 'google/flan-t5-base'

OUT_DIR      = '/kaggle/working/multi_engine_t5_model'

# ── Hyperparameters (Optimized for Single GPU P100) ──────────────────────
EPOCHS          = 3
BATCH_SIZE      = 4      # Safe for 512 input tokens
GRAD_ACCUM      = 4      # Effective batch size = 4 * 4 = 16
MAX_INPUT_LEN   = 512
MAX_TARGET_LEN  = 256
LEARNING_RATE   = 3e-4
WARMUP_STEPS    = 500

print(f'Train: {TRAIN_FILE}')
print(f'Val:   {VAL_FILE}')
print(f'Base model: {BASE_MODEL}')
print(f'Output: {OUT_DIR}')

assert os.path.isfile(TRAIN_FILE), f'MISSING: {TRAIN_FILE} — did you upload the dataset?'
assert os.path.isfile(VAL_FILE),   f'MISSING: {VAL_FILE}'
print('\nAll paths OK ✓')


In [ ]:
import json

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

print('Loading data...')
train_rows = load_jsonl(TRAIN_FILE)
val_rows   = load_jsonl(VAL_FILE)

# Engine distribution
engine_counts = {}
for r in train_rows:
    e = r.get('engine', 'oracle')
    engine_counts[e] = engine_counts.get(e, 0) + 1

print(f'Train: {len(train_rows):,} examples')
print(f'Val:   {len(val_rows):,} examples')
print('Engine distribution (train):')
for e, n in sorted(engine_counts.items()):
    print(f'  {e}: {n:,} ({n/len(train_rows)*100:.1f}%)')

# Peek at one example
ex = train_rows[0]
print('\nSample example keys:', list(ex.keys()))
print('instruction:', ex['instruction'][:120])
print('input:      ', ex['input'][:120])
print('output:     ', ex['output'][:120])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'\nLoading tokenizer and model from {BASE_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model     = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
from datasets import Dataset

def tokenize_batch(batch):
    # Combine instruction + input as the model's input text
    inputs = [
        f"{inst}\n\n{ctx}"
        for inst, ctx in zip(batch['instruction'], batch['input'])
    ]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        padding=False,
        truncation=True,
    )
    targets = tokenizer(
        text_target=batch['output'],
        max_length=MAX_TARGET_LEN,
        padding=False,
        truncation=True,
    )
    model_inputs['labels'] = targets['input_ids']
    return model_inputs

train_ds = Dataset.from_list(train_rows)
val_ds   = Dataset.from_list(val_rows)

print('Tokenizing train dataset...')
train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
print('Tokenizing val dataset...')
val_tok   = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)

print(f'Train tokenized: {len(train_tok):,} examples')
print(f'Val tokenized:   {len(val_tok):,} examples')
print('\nSample tokenized example:')
print('input_ids len:', len(train_tok[0]['input_ids']))
print('labels len:   ', len(train_tok[0]['labels']))


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f'GPU VRAM before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated / {torch.cuda.memory_reserved() / 1e9:.2f} GB reserved')

from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

# Dynamic padding — avoids batch shape errors
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    predict_with_generate=False,     # Fast epoch evaluation (loss-based)
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
    gradient_checkpointing=False,    # ~30% faster on P100
    dataloader_num_workers=4,        # Fast parallel data loading
    dataloader_pin_memory=True,
    logging_steps=100,
    report_to='none',
    save_total_limit=2,
)

print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Steps per epoch: {len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)}')

try:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
except TypeError:
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

print('\nStarting training...')
trainer.train()


In [ ]:
import os

print(f'Saving best model to {OUT_DIR}...')
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

# List saved files
files = os.listdir(OUT_DIR)
print(f'Saved {len(files)} files:')
for f in sorted(files):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f}  ({size/1e6:.1f} MB)')

print('\nDone! Download the output folder from: Output > multi_engine_t5_model/')

In [ ]:
# ── Quick sanity test — run a few inference examples ─────────────────────
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

test_tok = AutoTokenizer.from_pretrained(OUT_DIR)
test_mod = AutoModelForSeq2SeqLM.from_pretrained(OUT_DIR).to(device)
test_mod.eval()

test_cases = [
    {
        'engine': 'Oracle',
        'prompt': 'Analyze this Oracle database log entry and explain what went wrong, the likely cause, and how to fix it.\n\nORA-01555: snapshot too old: rollback segment number 4 with name "RBS4" too small'
    },
    {
        'engine': 'PostgreSQL',
        'prompt': 'Analyze this PostgreSQL database log entry and explain what went wrong, the likely cause, and how to fix it.\n\n2024-01-15 10:23:45 UTC [12345] ERROR:  duplicate key value violates unique constraint "users_pkey" (SQLSTATE 23505)'
    },
    {
        'engine': 'MySQL',
        'prompt': 'Analyze this MySQL database log entry and explain what went wrong, the likely cause, and how to fix it.\n\n2024-01-15T10:23:45.123456Z 5 [ERROR] [MY-001213] [Server] Deadlock found when trying to get lock; try restarting transaction'
    },
]

print('=== Inference Sanity Check ===\n')
for tc in test_cases:
    inputs = test_tok(tc['prompt'], return_tensors='pt').to(device)
    outputs = test_mod.generate(**inputs, max_new_tokens=150, num_beams=4)
    result = test_tok.decode(outputs[0], skip_special_tokens=True)
    print(f"[{tc['engine']}]")
    print(f"Output: {result}")
    print()

print('All done! Model is working correctly.')